# Docent DQL Notebook
Fetches agent runs via DQL and loads them into pandas.

### README
- Requires `pip install docent-python pandas`
- API key is prefilled; rotate in Docent if you regenerate keys.
- Run the cells as-is to fetch data into pandas.


In [ ]:
%pip install -q docent-python pandas


In [ ]:
from docent import Docent
import pandas as pd

API_KEY = "dk_Eugg9pVVozkL6Q6m_LjhOzT00ks7f2xZNNKo2YjzQGAavJrzLhOOMOSVtOscj1W"
SERVER_URL = "http://localhost:8890"
COLLECTION_ID = "7d99d911-8bfd-44f1-bb06-ca4ad59c5c3e"

DQL_QUERY = """
SELECT
  ar.id AS agent_run_id,
  ar.metadata_json->'agent'->>'model_name' AS metadata_agent_model_name,
  ar.metadata_json->'agent'->>'name' AS metadata_agent_name,
  ar.metadata_json->'scores'->>'reward' AS metadata_scores_reward,
  ar.metadata_json->>'task' AS metadata_task,
  ar.created_at AS created_at
FROM agent_runs ar
ORDER BY ar.created_at ASC
"""

client = Docent(api_key=API_KEY, server_url=SERVER_URL)
result = client.execute_dql(COLLECTION_ID, DQL_QUERY)
df = client.dql_result_to_df_experimental(result)

df.head()


In [ ]:
df_clean = df.dropna(subset=['metadata_scores_reward', 'metadata_task'])
df_clean.head()


In [ ]:
# Convert metadata_scores_reward to numeric
df_clean['metadata_scores_reward'] = pd.to_numeric(df_clean['metadata_scores_reward'])

# Create pivot table: average reward by task and model
pivot_table = df_clean.pivot_table(
    values='metadata_scores_reward',
    index='metadata_task',
    columns='metadata_agent_model_name',
    aggfunc='mean'
)

pivot_table
    

In [ ]:
import matplotlib.pyplot as plt

# Calculate the difference (5.1 - 5) for each task
diff = pivot_table['openai/gpt-5.1-codex'] - pivot_table['openai/gpt-5-codex']

# Plot histogram
plt.figure(figsize=(10, 6))
plt.hist(diff.dropna(), bins=20, edgecolor='black')
plt.xlabel('Score Difference (GPT-5.1 - GPT-5)')
plt.ylabel('Number of Tasks')
plt.title('Distribution of Performance Change (GPT-5.1 vs GPT-5)')
plt.axvline(x=0, color='red', linestyle='--', label='No change')
plt.legend()
plt.tight_layout()
plt.show()

# Print summary stats
print(f"Mean difference: {diff.mean():.4f}")
print(f"Median difference: {diff.median():.4f}")
print(f"Tasks where 5.1 is worse: {(diff < 0).sum()}")
print(f"Tasks where 5.1 is better: {(diff > 0).sum()}")
print(f"Tasks with no change: {(diff == 0).sum()}")


In [ ]:
# Get tasks where GPT-5.1 did worse than GPT-5
tasks_5_1_worse = diff[diff < 0].index.tolist()
print("Tasks where GPT-5.1 performed worse than GPT-5:")
tasks_5_1_worse


In [ ]:
# Filter df to agent runs where task is in the worse-performing list and model is gpt-5.1-codex
df_5_1_worse = df[
    (df['metadata_task'].isin(tasks_5_1_worse)) & 
    (df['metadata_agent_model_name'] == 'openai/gpt-5.1-codex')
]
df_5_1_worse


In [ ]:
for agent_run_id in df_5_1_worse['agent_run_id'].tolist():
    client.tag_transcript(COLLECTION_ID, agent_run_id, '51-worse')